In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,roc_auc_score,make_scorer,classification_report,ConfusionMatrixDisplay
import pickle


In [2]:
customers_df_cleaned=pd.read_csv(r"D:\VS_CODE\Rapido\drive-download-20260323T152350Z-3-001\cleaned_csv_files\customers.csv")
drivers_df_cleaned=pd.read_csv(r"D:\VS_CODE\Rapido\drive-download-20260323T152350Z-3-001\cleaned_csv_files\drivers.csv")
bookings_df_cleaned=pd.read_csv(r"D:\VS_CODE\Rapido\drive-download-20260323T152350Z-3-001\cleaned_csv_files\bookings.csv")

In [3]:
#Create a new df for Ride Outcome prediction model
customers_mod_df=customers_df_cleaned[['customer_id','loyalty_score']].rename(columns={'loyalty_score':'customer_loyalty_score'})
drivers_mod_df=drivers_df_cleaned[['driver_id','reliability_score']].rename(columns={'reliability_score':'driver_reliability_score'})


print(customers_mod_df.head(2))
print(drivers_mod_df.head(2))

  customer_id  customer_loyalty_score
0    C_000001                   97.78
1    C_000002                  102.44
  driver_id  driver_reliability_score
0  D_000001                  0.710000
1  D_000002                  0.717143


In [4]:
bookings_df_mod=bookings_df_cleaned.merge(customers_mod_df,on='customer_id',how='left')
bookings_df_mod.head(2)

,booking_id,booking_date,booking_time,day_of_week,is_weekend,hour_of_day,city,pickup_location,drop_location,vehicle_type,...,booking_status,incomplete_ride_reason,customer_id,driver_id,fare_per_km,fare_per_min,rush_hour_flag,long_distance_flag,city_pair,customer_loyalty_score
0,B_000001,2025-12-11,00:07:00,Thursday,0,0,Mumbai,Loc_19,Loc_16,Bike,...,Cancelled,Others,C_005097,D_004592,21.144080,3.201296,0,0,Loc_19 Loc_16,96.15
1,B_000002,2025-07-07,06:13:00,Monday,0,6,Mumbai,Loc_32,Loc_38,Cab,...,Completed,Others,C_008459,D_000148,48.174767,11.018212,0,0,Loc_32 Loc_38,108.85


In [5]:
bookings_df_mod=bookings_df_mod.merge(drivers_mod_df,on='driver_id',how='left')
bookings_df_mod.head(2)

,booking_id,booking_date,booking_time,day_of_week,is_weekend,hour_of_day,city,pickup_location,drop_location,vehicle_type,...,incomplete_ride_reason,customer_id,driver_id,fare_per_km,fare_per_min,rush_hour_flag,long_distance_flag,city_pair,customer_loyalty_score,driver_reliability_score
0,B_000001,2025-12-11,00:07:00,Thursday,0,0,Mumbai,Loc_19,Loc_16,Bike,...,Others,C_005097,D_004592,21.144080,3.201296,0,0,Loc_19 Loc_16,96.15,0.759778
1,B_000002,2025-07-07,06:13:00,Monday,0,6,Mumbai,Loc_32,Loc_38,Cab,...,Others,C_008459,D_000148,48.174767,11.018212,0,0,Loc_32 Loc_38,108.85,0.730667


In [6]:
bookings_df_mod.columns

Index(['booking_id', 'booking_date', 'booking_time', 'day_of_week',
       'is_weekend', 'hour_of_day', 'city', 'pickup_location', 'drop_location',
       'vehicle_type', 'ride_distance_km', 'estimated_ride_time_min',
       'actual_ride_time_min', 'traffic_level', 'weather_condition',
       'base_fare', 'surge_multiplier', 'booking_value', 'booking_status',
       'incomplete_ride_reason', 'customer_id', 'driver_id', 'fare_per_km',
       'fare_per_min', 'rush_hour_flag', 'long_distance_flag', 'city_pair',
       'customer_loyalty_score', 'driver_reliability_score'],
      dtype='object')

In [7]:
bookings_df_mod=bookings_df_mod.drop(['booking_id', 'booking_date', 'booking_time','actual_ride_time_min','fare_per_km','fare_per_min','city_pair','incomplete_ride_reason'],axis=1)
bookings_df_mod.columns

Index(['day_of_week', 'is_weekend', 'hour_of_day', 'city', 'pickup_location',
       'drop_location', 'vehicle_type', 'ride_distance_km',
       'estimated_ride_time_min', 'traffic_level', 'weather_condition',
       'base_fare', 'surge_multiplier', 'booking_value', 'booking_status',
       'customer_id', 'driver_id', 'rush_hour_flag', 'long_distance_flag',
       'customer_loyalty_score', 'driver_reliability_score'],
      dtype='object')

In [8]:
bookings_df_mod=bookings_df_mod.reindex(columns=['day_of_week', 'is_weekend', 'hour_of_day', 'city', 'pickup_location',
       'drop_location', 'vehicle_type', 'ride_distance_km',
       'estimated_ride_time_min', 'traffic_level', 'weather_condition',
       'base_fare', 'surge_multiplier', 'booking_value', 
        'customer_id', 'driver_id', 'rush_hour_flag',
       'long_distance_flag', 'customer_loyalty_score',
       'driver_reliability_score','booking_status'])
bookings_df_mod.head(2)

,day_of_week,is_weekend,hour_of_day,city,pickup_location,drop_location,vehicle_type,ride_distance_km,estimated_ride_time_min,traffic_level,...,base_fare,surge_multiplier,booking_value,customer_id,driver_id,rush_hour_flag,long_distance_flag,customer_loyalty_score,driver_reliability_score,booking_status
0,Thursday,0,0,Mumbai,Loc_19,Loc_16,Bike,7.01,46.30,High,...,76.12,2.0,148.22,C_005097,D_004592,0,0,96.15,0.759778,Cancelled
1,Monday,0,6,Mumbai,Loc_32,Loc_38,Cab,9.67,43.54,Medium,...,254.15,1.8,465.85,C_008459,D_000148,0,0,108.85,0.730667,Completed


In [9]:
bookings_df_mod.columns

Index(['day_of_week', 'is_weekend', 'hour_of_day', 'city', 'pickup_location',
       'drop_location', 'vehicle_type', 'ride_distance_km',
       'estimated_ride_time_min', 'traffic_level', 'weather_condition',
       'base_fare', 'surge_multiplier', 'booking_value', 'customer_id',
       'driver_id', 'rush_hour_flag', 'long_distance_flag',
       'customer_loyalty_score', 'driver_reliability_score', 'booking_status'],
      dtype='object')

In [10]:
bookings_df_mod.dtypes

day_of_week                  object
is_weekend                    int64
hour_of_day                   int64
city                         object
pickup_location              object
drop_location                object
vehicle_type                 object
ride_distance_km            float64
estimated_ride_time_min     float64
traffic_level                object
weather_condition            object
base_fare                   float64
surge_multiplier            float64
booking_value               float64
customer_id                  object
driver_id                    object
rush_hour_flag                int64
long_distance_flag            int64
customer_loyalty_score      float64
driver_reliability_score    float64
booking_status               object
dtype: object

Change the object data types to string data types before building the catboost model

In [11]:
object_cols=bookings_df_mod.select_dtypes(include=["object"]).columns
bookings_df_mod[object_cols]=bookings_df_mod[object_cols].astype("string")
bookings_df_mod.dtypes

day_of_week                 string[python]
is_weekend                           int64
hour_of_day                          int64
city                        string[python]
pickup_location             string[python]
drop_location               string[python]
vehicle_type                string[python]
ride_distance_km                   float64
estimated_ride_time_min            float64
traffic_level               string[python]
weather_condition           string[python]
base_fare                          float64
surge_multiplier                   float64
booking_value                      float64
customer_id                 string[python]
driver_id                   string[python]
rush_hour_flag                       int64
long_distance_flag                   int64
customer_loyalty_score             float64
driver_reliability_score           float64
booking_status              string[python]
dtype: object

In [12]:
bookings_df_mod['booking_status'].unique()

<StringArray>
['Cancelled', 'Completed', 'Incomplete']
Length: 3, dtype: string

In [14]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)
import pandas as pd

# Target column
target = "booking_status"

# Features
X = bookings_df_mod.drop(columns=[target])
y = bookings_df_mod[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identify categorical features
categorical_features = X.select_dtypes(include=["string"]).columns.tolist()
#categorical_features = ['vehicle_type','city','traffic_level','weather_condition']

# Build CatBoost Pools
train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

# Initialize CatBoostClassifier
cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

# Train
cat_model.fit(train_pool, eval_set=test_pool)

# Predictions
y_train_pred = cat_model.predict(X_train)
y_test_pred = cat_model.predict(X_test)

# Probabilities (needed for AUC)
y_train_proba = cat_model.predict_proba(X_train)
y_test_proba = cat_model.predict_proba(X_test)

# --- Training Metrics ---
print("TRAINING METRICS")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("F1-score (macro):", f1_score(y_train, y_train_pred, average="macro"))
print("ROC-AUC (OvR, macro):", roc_auc_score(y_train, y_train_proba, multi_class="ovr", average="macro"))
print("\nClassification Report:\n", classification_report(y_train, y_train_pred))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))

# --- Test Metrics ---
print("\nTEST METRICS")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1-score (macro):", f1_score(y_test, y_test_pred, average="macro"))
print("ROC-AUC (OvR, macro):", roc_auc_score(y_test, y_test_proba, multi_class="ovr", average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


0:	learn: 0.6995250	test: 0.6998000	best: 0.6998000 (0)	total: 249ms	remaining: 2m 4s
100:	learn: 0.7202750	test: 0.7292500	best: 0.7296000 (96)	total: 17.2s	remaining: 1m 8s
200:	learn: 0.7251750	test: 0.7352000	best: 0.7352000 (200)	total: 41.2s	remaining: 1m 1s
300:	learn: 0.7307000	test: 0.7363000	best: 0.7364500 (266)	total: 1m 7s	remaining: 44.7s
400:	learn: 0.7357625	test: 0.7365000	best: 0.7369500 (362)	total: 1m 33s	remaining: 23.2s
499:	learn: 0.7402875	test: 0.7370500	best: 0.7372500 (470)	total: 2m 3s	remaining: 0us

bestTest = 0.73725
bestIteration = 470

Shrink model to first 471 iterations.
TRAINING METRICS
Accuracy: 0.685825
F1-score (macro): 0.3895577166947141
ROC-AUC (OvR, macro): 0.7079580845423016

Classification Report:
               precision    recall  f1-score   support

   Cancelled       0.43      0.26      0.33     18627
   Completed       0.73      0.91      0.81     54677
  Incomplete       0.42      0.02      0.03      6696

    accuracy                  

In [17]:

with open ("cat_model_ride_outcome_pred.pkl","wb") as file:
    pickle.dump(cat_model,file)